# Instagram two-agent pipeline (Agent1 unchanged) + Tavily evidence + improved Agent2

- Agent1: splits text into **Discriminative Text** and **Residual Text** (kept identical to your previous logic).
- Tavily: searches for **contact/company legitimacy + scam reports** for discriminative samples.
- Agent2: makes final label using **(a) agent1 output + (b) Tavily evidence**.

> **Label convention used here (based on your `instagram_final_results` errors):**
> - **1 = benign / normal profile**
> - **0 = suspicious / potential scam / unverifiable contact**

Set environment variables before running:
- `OPENAI_API_KEY`, `OPENAI_BASE_URL` (optional), `TAVILY_API_KEY`


In [ ]:

import os
import re
import time
import json
from typing import List, Dict, Any, Tuple

import requests
import pandas as pd
from openai import OpenAI
import re
from typing import Dict


# 0) Keys / Clients

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "Input your key")
OPENAI_BASE_URL = os.getenv("OPENAI_API_BASE", os.getenv("OPENAI_BASE_URL", "Input your URL"))

# Tavily
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY", "Input your key")
TAVILY_ENDPOINT = os.getenv("TAVILY_ENDPOINT", "Input your URL")

if not OPENAI_API_KEY:
    raise RuntimeError("Missing OPENAI_API_KEY env var")

client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)


def retry_with_backoff(func, *args, retries=10, delay=2, **kwargs):
    """Simple exponential backoff retry."""
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(delay)
                delay *= 2
            else:
                raise last_err


def response_API(user_prompt: str, model: str = "gpt-4o-mini", temperature: float = 0.0) -> str:
    """Call LLM (single-user message, compatible with your original code)."""

    def api_call():
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": user_prompt}],
            temperature=temperature,
        )
        return resp.choices[0].message.content

    return retry_with_backoff(api_call, retries=8, delay=2)



# 1) Agent1 prompt (UNCHANGED)

AGENT1_PROMPT = r"""

You are Agent1.

You must extract the parts of a Reddit comment that could influence the final label.
This is a LENIENT extraction task: when in doubt, INCLUDE it in Discriminative Text.


You must copy exact original text (do NOT paraphrase). Include ALL lines/fragments that match ANY of the following:

(1) URLs / links / domains / emails / @handles
- Any "http", "https", "www", ".com", ".org", ".net", short links, YouTube links, news links, etc.

(2) Negative or toxic language (very common in label=0)
- Profanity, insults, slurs, name-calling, demeaning terms
- Strong negative judgments: "bullshit", "shit show", "dipshit", "idiot", "stupid", "garbage", etc.
- Mocking sarcasm that targets a person/group

(3) Strong accusations / conspiracy / criminal claims / extreme claims
- Claims like: "criminal enterprise", "they will kill people", "corruption", "steal organs", "cartel", "murder", etc.
- “They are hiding”, “planned”, “controlled by”, “they don’t want you to know”
- Absolute certainty without evidence: "always", "never", "100%", "definitely"

(4) Political / institutional / media entities or public figures (often decisive in your dataset)
- Names like: NHS, Daily Mail, government, taxpayers, Ford, Kavanaugh, etc.
- Any party, country, institution, newspaper, politician, police, court, etc.
Include the sentence where the name appears (full sentence, not a fragment).

(5) Capitalization emphasis and notable names (your requirement)
- ALL CAPS words, SHOUTING, repeated exclamation marks
- Proper nouns / capitalized names of people/places/organizations
- If a capitalized name appears, include the surrounding sentence.

(6) Money / numbers / dates / statistics (often used as “evidence” in arguments)
- $12, $15, £300,000, 2020, percentages, quantities, "years", "FPS", hardware specs if argued as proof
Include the sentence containing these numbers.

(7) Calls to action / persuasion / urging behavior
- "DM me", "message me", "join", "click", "download", "test it yourself", "act now"
Include the entire call-to-action line.

(8) Quoted text (Reddit ">") handling
- If the comment contains quoted lines starting with ">", include:
  a) the quoted line IF the author agrees with it or uses it as support, OR
  b) the author’s immediate reply that clarifies stance.
When uncertain, include BOTH the quote and the author reply (lenient).


If the comment is truly neutral (normal discussion/joke/technical talk with no negativity, no links, no accusations):
- Put the single most representative sentence into Discriminative Text.

Now process the given Reddit comment.

EXTRACTION STRATEGY

- Discriminative Text should contain ALL sensitive or high-risk signals.
- If multiple risky elements exist, include ALL of them.
- Do NOT summarize.
- Copy exact wording from original comment.
- Do NOT remove strong adjectives or offensive words — keep them.


CRITICAL FORMAT LOCK:

Your entire output must contain EXACTLY two lines.
Line 1 must start with:
Discriminative Text:

Line 2 must start with:
Residual Text:

There must be NO blank lines.
There must be NO extra spaces before the labels.
There must be NO text before or after the two lines.


""".strip()



DISC_RE = re.compile(r"^\s*Discriminative\s*Text\s*:?\s*(.*)\s*$", re.IGNORECASE)
RESI_RE = re.compile(r"^\s*Residual\s*Text\s*:?\s*(.*)\s*$", re.IGNORECASE)

def agent1_process(original_text) -> Dict[str, str]:
    if not isinstance(original_text, str):
        original_text = ""

    full_prompt = AGENT1_PROMPT + "\n\n" + original_text

    try:
        response = response_API(full_prompt)
    except Exception as e:
        print("Agent1 API failed:", e)
        response = ""

    if not response.strip():
        return {"Discriminative Text": "", "Residual Text": original_text or ""}

    disc_lines, resid_lines = [], []
    current_section = None

    for raw_line in response.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        m_d = DISC_RE.match(line)
        if m_d:
            current_section = "D"
            # Critical fix: The content after the colon on the same line should also be included.
            inline = m_d.group(1).strip()
            if inline:
                disc_lines.append(inline)
            continue

        m_r = RESI_RE.match(line)
        if m_r:
            current_section = "R"
            inline = m_r.group(1).strip()
            if inline:
                resid_lines.append(inline)
            continue

        if current_section == "D":
            disc_lines.append(line)
        elif current_section == "R":
            resid_lines.append(line)

    # Backup: If Agent1 outputs data completely out of format, treat all instances as residuals to avoid data loss.
    disc_text = "\n".join(disc_lines).strip()
    resid_text = "\n".join(resid_lines).strip()
    if not disc_text and not resid_text:
        return {"Discriminative Text": "", "Residual Text": original_text or ""}

    # As a fallback: If the residual is empty, but the original text exists, put the original text into the residual.
    if not resid_text and original_text:
        resid_text = original_text

    return {
        "Discriminative Text": disc_text,
        "Residual Text": resid_text,
    }


def run_agent1_and_save_csv(input_csv: str, output_csv: str, sleep_s: float = 0.5):
    df = pd.read_csv(input_csv)

    disc_col, resid_col = [], []
    for idx, row in df.iterrows():
        text = row.get("text", "")
        if not isinstance(text, str):
            text = ""  # Prevent NaN

        result = agent1_process(text)
        disc_col.append(result.get("Discriminative Text", ""))
        resid_col.append(result.get("Residual Text", text))

        if idx % 10 == 0:
            print(f"Agent1 processed {idx} samples")
        time.sleep(sleep_s)

    df["Discriminative Text"] = disc_col
    df["Residual Text"] = resid_col
    df.to_csv(output_csv, index=False)
    print("Agent1 CSV saved to:", output_csv)



# 2) Tavily search (pattern copied from your combined_script style)


def tavily_search(query: str, retries: int = 3, delay: int = 2, max_results: int = 2, search_depth: str = "basic") -> Dict[str, Any]:
    if not TAVILY_API_KEY:
        raise RuntimeError("Missing TAVILY_API_KEY env var")

    headers = {"Content-Type": "application/json"}
    payload = {
        "api_key": TAVILY_API_KEY,
        "query": query,
        "max_results": max_results,
        "search_depth": search_depth,
        "include_answer": False,
        "include_raw_content": False,
    }

    def api_call():
        resp = requests.post(TAVILY_ENDPOINT, json=payload, headers=headers, timeout=30)
        resp.raise_for_status()
        return resp.json()

    return retry_with_backoff(api_call, retries=retries, delay=delay)


EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
PHONE_RE = re.compile(r"(?:(?:\+|00)\d{1,3}[\s-]?)?(?:\(?\d{2,4}\)?[\s-]?)?\d{3,4}[\s-]?\d{3,4}")
HANDLE_RE = re.compile(r"(?<!\w)@[A-Za-z0-9._]{2,30}")
URL_RE = re.compile(r"https?://\S+|\b(?:[A-Za-z0-9-]+\.)+[A-Za-z]{2,}\b")


def extract_search_targets(discriminative_text: str) -> List[str]:
    """Extract emails/phones/handles/domains to look up."""
    disc = discriminative_text or ""
    targets = []

    targets += EMAIL_RE.findall(disc)

    # Phones: filter out very short matches that are likely dates or counts
    for m in PHONE_RE.findall(disc):
        digits = re.sub(r"\D", "", m)
        if len(digits) >= 9 and len(digits) <= 15:
            targets.append(m.strip())

    targets += HANDLE_RE.findall(disc)

    # domains/urls
    for m in URL_RE.findall(disc):
        m = m.strip().strip(".,;:!()[]{}<>")
        if m and m not in targets:
            targets.append(m)

    # de-dup preserve order
    seen = set()
    out = []
    for t in targets:
        if t.lower() not in seen:
            seen.add(t.lower())
            out.append(t)
    return out[:6]  # cap


def tavily_evidence_for_row(row: pd.Series) -> Dict[str, Any]:
    """
    Only search when Agent1 produced valid discriminative text.
    If discriminative text is empty / NaN / meaningless → skip Tavily.
    """

    disc = row.get("Discriminative Text", None)

    # Bypass external search if the input is completely empty or null
    if disc is None:
        return {"targets": [], "queries": [], "results": []}

    if not isinstance(disc, str):
        return {"targets": [], "queries": [], "results": []}

    disc = disc.strip()
    if disc == "" or disc.lower() == "nan":
        return {"targets": [], "queries": [], "results": []}

    # Trigger the retrieval logic strictly for valid, substantive content
    targets = extract_search_targets(disc)
    if not targets:
        return {"targets": [], "queries": [], "results": []}

    queries = []
    results_compact = []

    for t in targets:
        q = f"{t} scam OR fraud OR complaint OR review"
        queries.append(q)

        try:
            res = tavily_search(q, max_results=2, search_depth="basic")
        except Exception as e:
            res = {"error": str(e), "results": []}

        compact_items = []
        for item in (res.get("results") or [])[:2]:
            compact_items.append({
                "title": item.get("title", ""),
                "url": item.get("url", ""),
                "content": item.get("content", "")[:280],
                "score": item.get("score", None),
            })

        results_compact.append({
            "target": t,
            "query": q,
            "items": compact_items
        })

        time.sleep(0.2)

    return {
        "targets": targets,
        "queries": queries,
        "results": results_compact
    }



def add_tavily_evidence_column(agent1_csv: str, output_csv: str, sleep_s: float = 0.2):
    df = pd.read_csv(agent1_csv)

    evidences = []
    for idx, row in df.iterrows():
        ev = tavily_evidence_for_row(row)
        evidences.append(json.dumps(ev, ensure_ascii=False))
        if idx % 10 == 0:
            print(f"Tavily processed {idx} samples")
        time.sleep(sleep_s)

    df["tavily_evidence"] = evidences
    df.to_csv(output_csv, index=False)
    print("Agent1+Tavily CSV saved to:", output_csv)



# 3) Improved Agent2 prompt (fix label direction + reduce false positives)


AGENT2_PROMPT = r"""
You are Agent2. You must output ONLY a single character: 0 or 1.

Label definition (Reddit-adapted):
1 = benign / normal Reddit discussion (including disagreement and mild negativity)
0 = suspicious / harmful / toxic / manipulative / high-risk unverifiable

You will receive:
- Discriminative Text (PRIMARY evidence, highest weight)
- Residual Text (SECONDARY evidence, important context; do not ignore)
- Tavily evidence JSON (TERTIARY reference only; may be empty for Reddit comments)

WEIGHTED DECISION:
1) Start from Discriminative Text.
2) Use Residual Text to confirm missed triggers and context.
3) Use Tavily only to strengthen confidence when available; if empty/irrelevant, ignore it.

IMPORTANT:
- Do NOT label 0 just because the comment is negative or political.
- However, Reddit label=0 in this dataset often includes toxic/hostile harassment, extreme punishment advocacy, and incitement.

---------------------------------------
Assign label = 0 if ANY of the following is present
(in Discriminative Text OR Residual Text):
---------------------------------------

A) Targeted hostility / harassment (lower threshold)
- Direct second-person attack or belittling ("you are...", "you sound like...", "you don't know...", etc.)
- Name-calling, insults, demeaning language aimed at a person (not just an idea)
- Aggressive profanity used to attack/condemn a person or group (esp. with "you/they")
- Mocking sarcasm that targets a person/group rather than discussing the topic

B) Extreme harm / punishment / incitement
- Calls for extreme punishment ("imprison them for life", "they deserve to die", etc.)
- Encouraging violence or illegal actions, even implicitly

C) Hate / dehumanization
- Slurs, dehumanizing labels, or blanket hatred toward a group

D) Manipulation / scam / action-driving behavior
- DM/contact privately, Telegram/WhatsApp/Discord
- guaranteed profit / pumping
- urgency pressure ("act now")
- suspicious link used to lure/recruit/extract money/info

E) High-risk unverifiable factual certainty claims
- serious criminal accusations stated as fact without support
- conspiracy claims presented as certainty
- strong medical/legal/financial advice stated as fact without evidence

---------------------------------------
Assign label = 1 if:
---------------------------------------
- It is normal discussion/opinion/argument/complaint, even if strongly negative,
  AS LONG AS it is not targeted harassment and not advocating harm/punishment.
- It criticizes politics/media/institutions without personal attacks or harm advocacy.
- Tavily is empty and there is no clear 0-trigger -> output 1.

Tie-break:
- If there is any clear targeted harassment OR harm/punishment advocacy -> output 0.
- If uncertain and no clear trigger -> output 1.

OUTPUT RULE:
Return ONLY "0" or "1". No other text.


""".strip()


def agent2_judge_from_row(row: pd.Series, sleep_s: float = 0.0) -> int:
    disc = row.get("Discriminative Text", "")
    resid = row.get("Residual Text", "")
    tav = row.get("tavily_evidence", "")

    # If Agent1 found nothing discriminative, default to benign.
    if not isinstance(disc, str) or not disc.strip():
        return 1

    user_input = f"""
Discriminative Text:\n{disc}

Residual Text:\n{resid}

Tavily evidence JSON:\n{tav}
""".strip()

    resp = response_API(AGENT2_PROMPT + "\n\n" + user_input)
    time.sleep(sleep_s)


def run_agent2_and_save_csv(agent1_tavily_csv: str, output_csv: str, sleep_s: float = 0.5):
    df = pd.read_csv(agent1_tavily_csv)

    preds = []
    for idx, row in df.iterrows():
        disc = row.get("Discriminative Text", "")
        resid = row.get("Residual Text", "")
        tav = row.get("tavily_evidence", "")

        if not isinstance(disc, str): disc = ""
        if not isinstance(resid, str): resid = ""
        if not isinstance(tav, str): tav = ""

        user_input = (
            "Discriminative Text:\n" + disc +
            "\n\nResidual Text:\n" + resid +
            "\n\nTavily Evidence:\n" + tav
        )

        full_prompt = AGENT2_PROMPT + "\n\n" + user_input

        try:
            resp = response_API(full_prompt)
        except Exception as e:
            print("Agent2 API failed:", e)
            resp = ""

        # Stable analysis: First 0/1 caught
        m = re.search(r"[01]", str(resp))
        pred = int(m.group(0)) if m else 1
        preds.append(pred)

        if idx % 10 == 0:
            print(f"Agent2 processed {idx} samples")
        time.sleep(sleep_s)

    df["agent2_pred"] = preds
    df.to_csv(output_csv, index=False)
    print("Agent2 CSV saved to:", output_csv)



# 4) Main (Colab-friendly paths)

if __name__ == "__main__":
    # Update these paths as needed
    input_csv = "/content/reddit_samples200.csv"
    after_a1 = "/content/reddit_after_agent1.csv"
    after_tav = "/content/reddit_after_tavily.csv"
    final_out = "/content/reddit_final_results.csv"

    run_agent1_and_save_csv(input_csv=input_csv, output_csv=after_a1)
    add_tavily_evidence_column(agent1_csv=after_a1, output_csv=after_tav)
    run_agent2_and_save_csv(agent1_tavily_csv=after_tav, output_csv=final_out)


Agent1 processed 0 samples
Agent1 processed 10 samples
Agent1 processed 20 samples
Agent1 processed 30 samples
Agent1 processed 40 samples
Agent1 processed 50 samples
Agent1 processed 60 samples
Agent1 processed 70 samples
Agent1 processed 80 samples
Agent1 processed 90 samples
Agent1 processed 100 samples
Agent1 processed 110 samples
Agent1 processed 120 samples
Agent1 processed 130 samples
Agent1 processed 140 samples
Agent1 processed 150 samples
Agent1 processed 160 samples
Agent1 processed 170 samples
Agent1 processed 180 samples
Agent1 processed 190 samples
Agent1 CSV saved to: /content/reddit_after_agent1.csv
Tavily processed 0 samples
Tavily processed 10 samples
Tavily processed 20 samples
Tavily processed 30 samples
Tavily processed 40 samples
Tavily processed 50 samples
Tavily processed 60 samples
Tavily processed 70 samples
Tavily processed 80 samples
Tavily processed 90 samples
Tavily processed 100 samples
Tavily processed 110 samples
Tavily processed 120 samples
Tavily proc

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# Refactored to use a relative path to ensure cross-platform compatibility
df_result = pd.read_csv("/content/reddit_final_results.csv")

y_true = df_result["label"]
y_pred = df_result["agent2_pred"]

# Retain only rows where both ground truth labels and predictions are valid (non-null)
mask = y_true.notna() & y_pred.notna()

y_true_clean = y_true[mask]
y_pred_clean = y_pred[mask]

# Calculate core binary classification metrics
acc = accuracy_score(y_true_clean, y_pred_clean)
p   = precision_score(y_true_clean, y_pred_clean, average="binary")
r   = recall_score(y_true_clean, y_pred_clean, average="binary")
f1  = f1_score(y_true_clean, y_pred_clean, average="binary")

#F1-macro
f1_macro = f1_score(y_true_clean, y_pred_clean, average="macro")

print("Used rows:", mask.sum(), "/", len(df_result))
print("Accuracy:", acc,
      "Precision:", p,
      "Recall:", r,
      "F1:", f1,
      "F1-macro:", f1_macro)


Used rows: 200 / 200
Accuracy: 0.565 Precision: 0.554140127388535 Recall: 0.8365384615384616 F1: 0.6666666666666666 F1-macro: 0.5203836930455635
